In [36]:
import pandas as pd
import folium
import matplotlib.colors as mcolors
import re


# CSV 파일 경로
file_path = "C:/Users/User/Desktop/VSCODE/Data_Analysis_/envoir/국토교통부_쓰레기 무단투기 상습 다발지역 분석_시각화_20210101 (1).csv"

# 인코딩 및 구분자 설정 (euc-kr 인코딩, 쉼표 구분자)
df = pd.read_csv(file_path, encoding='euc-kr')

# 필요한 열만 선택
df = df[["지역아이디", "상습지역지수합계", "상습지역지수평균", "다발지역지수", "지오메트리"]]

# 확인
print(df.head())



                                                                                                   지역아이디  \
50f913f9-28-20211104180343-481 13.333333 13.333333 14  MULTIPOLYGON (((126.815457095772 37.5707882352037   
80f913f9-5-20211104180324-495  13.333333 13.333333 20   MULTIPOLYGON (((126.81488679108 37.5712355773762   
40f913f9-1-20211104180323-39   16.129032 16.129032 14  MULTIPOLYGON (((126.832610995896 37.5519582897201   
70f913f9-15-20211104180334-442 41.935484 41.935484 14  MULTIPOLYGON (((126.808084373739 37.5720972789322   
00f913f9-10-20211104180329-36  16.129032 16.129032 15  MULTIPOLYGON (((126.833181049131 37.5515108637286   

                                                                                상습지역지수합계  \
50f913f9-28-20211104180343-481 13.333333 13.333333 14  126.815461119085 37.5703375821795   
80f913f9-5-20211104180324-495  13.333333 13.333333 20  126.814890814393 37.5707849662615   
40f913f9-1-20211104180323-39   16.129032 16.129032 14  126.832615103029 37.

In [37]:
def extract_coords(geom_str):
    try:
        if not geom_str.endswith(")))"):
            geom_str += ")))"
        coords = re.findall(r"([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)", geom_str)
        return [(float(x), float(y)) for x, y in coords]
    except:
        return []

df["coords"] = df["지오메트리"].apply(extract_coords)


In [38]:
df["centroid_lon"] = df["coords"].apply(lambda c: sum(x for x, _ in c) / len(c) if c else None)
df["centroid_lat"] = df["coords"].apply(lambda c: sum(y for _, y in c) / len(c) if c else None)
df = df.dropna(subset=["centroid_lat", "centroid_lon", "상습지역지수합계", "다발지역지수"])
df["sum"] = df["상습지역지수합계"].astype(str).str.strip().str.split().str[0]
df["count"] = pd.to_numeric(df["다발지역지수"], errors="coerce")
df["sum"] = pd.to_numeric(df["sum"], errors="coerce")


상습지역지수합계

In [39]:
m = folium.Map(tiles="cartodbpositron")
norm = mcolors.Normalize(vmin=df["sum"].min(), vmax=df["sum"].max())

for _, row in df.iterrows():
    intensity = float(norm(row["sum"]))
    r, g, b = 1.0, max(0.0, 1.0 - intensity), max(0.0, 1.0 - intensity)
    color = mcolors.to_hex((r, g, b))

    folium.CircleMarker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>지역:</b> {row['지역아이디']}<br><b>지수합계:</b> {row['sum']}<br><b>다발지수:</b> {row['count']}",
            max_width=250
        )
    ).add_to(m)


In [40]:
m.fit_bounds(df[["centroid_lat", "centroid_lon"]].values.tolist())
m


In [41]:
# 상습지역지수합계 기준 내림차순 정렬
sorted_by_sum = df[['상습지역지수합계', '지오메트리']]
sorted_by_sum = sorted_by_sum.sort_values(by='상습지역지수합계', ascending=False)

# 결과 출력 (상위 10개) - 지오메트리만 출력
print(sorted_by_sum[['지오메트리']].head(10))


                                                                                      지오메트리
60f913f9-21-20211104180334-201 16.666667 16.666667 15  126.859233680581 37.5498524203663)))
b0f913f9-24-20211104180342-167 22.580645 22.580645 35  126.851184538962 37.5642299827022)))
20f913f9-1-20211104180323-183  13.333333 13.333333 10  126.850275353925 37.5385366834414)))
e0f913f9-3-20211104180322-160  12.903226 12.903226 24  126.848526553646 37.5443857850248)))
50f913f9-2-20211104180322-156  12.903226 12.903226 36  126.848522697971 37.5448363961395)))
60f913f9-0-20211104180322-164  12.903226 12.903226 16   126.84795658423 37.5448332529258)))
30f913f9-2-20211104180322-130  45.161290 45.161290 41  126.844612121044 37.5389560300573)))
20f913f9-27-20211104180343-143 45.161290 45.161290 17   126.84460818155 37.5394066830815)))
f0f913f9-22-20211104180344-107 45.161290 45.161290 21  126.844046174942 37.5389528868436)))
d0f913f9-3-20211104180322-102  23.333333 23.333333 54  126.840741861072 37.52856

In [53]:
import requests

# 카카오 API 키
KAKAO_REST_API_KEY = "a6c4ce897809e27dd0e9fad56832a958"

# 카카오 API를 통한 역지오코딩
def kakao_reverse_geocode(lat, lon):
    url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"}
    params = {"x": lon, "y": lat}  # 카카오는 (경도, 위도) 순서로 요청을 보냄
    res = requests.get(url, headers=headers, params=params)
    
    # 응답 상태 코드 확인
    if res.status_code != 200:
        print(f"API 요청 실패: {res.status_code}")
        return "주소 없음"
    
    # 응답 내용 확인
    try:
        documents = res.json().get("documents", [])
        if documents:
            return documents[0]["address"]["address_name"]
        else:
            print(f"주소를 찾을 수 없음 at ({lat}, {lon})")
            return "주소 없음"
    except Exception as e:
        print(f"오류 발생 at ({lat}, {lon}): {e}")
        return "주소 없음"

# 상습지역지수합계 기준 내림차순 정렬
sorted_by_sum = df[['상습지역지수합계', '지오메트리']]
sorted_by_sum = sorted_by_sum.sort_values(by='상습지역지수합계', ascending=False)

# 지오메트리에서 좌표 추출 후 카카오 API 호출하여 주소를 얻고 리스트에 저장
addresses_sum = []
for _, row in sorted_by_sum.iterrows():
    if row["지오메트리"]:
        coords = extract_coords(row["지오메트리"])
        lat, lon = coords[0]  # 첫 번째 좌표 사용
        address = kakao_reverse_geocode(lat, lon)
        addresses_sum.append(address)
    else:
        addresses_sum.append("주소 없음")

# 결과를 DataFrame에 저장
sorted_by_sum['주소'] = addresses_sum
print(sorted_by_sum[['지오메트리', '주소']].head(10))


API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
API 요청 실패: 403
                                                                                      지오메트리  \
60f913f9-21-20211104180334-201 16.666667 16.666667 15  126.859233680581 37.5498524203663)))   
b0f913f9-24-20211104180342-167 22.580645 22.580645 35  126.851184538962 37.5642299827022)))   
20f913f9-1-20211104180323-183  13.333333 13.333333 10  126.850275353925 37.5385366834414)))   
e0f913f9-3-20211104180322-160  12.903226 12.903226 24  126.848526553646 37.54438

다발지역지수별 표시

# 다발지역지수 값이 이상한 row만 출력
mask = df["다발지역지수"].astype(str).str.strip().replace("", pd.NA).isna()
print(df[mask][["지역아이디", "다발지역지수"]])
df["count"] = df["다발지역지수"].astype(str).str.strip().replace("", pd.NA)
df["count"] = pd.to_numeric(df["count"], errors="coerce")


In [43]:
# 공백 분리 후 첫 번째 값만 가져오기
df["count"] = df["다발지역지수"].astype(str).str.strip().str.split().str[0]
df["count"] = pd.to_numeric(df["count"], errors="coerce")


In [44]:
# 1. folium 지도 생성
m = folium.Map(tiles="cartodbpositron")

# 2. 정규화 (다발지역지수 기준)
norm = mcolors.Normalize(vmin=df["count"].min(), vmax=df["count"].max())

# 3. 마커 추가 (좌표, count 결측치 제거)
for _, row in df.iterrows():
    # 좌표 없거나 count 비었으면 건너뜀
    if pd.isnull(row["count"]) or pd.isnull(row["centroid_lat"]) or pd.isnull(row["centroid_lon"]):
        continue

    intensity = float(norm(row["count"]))  # 0~1 사이 정규화
    r, g, b = 1.0 - intensity, 1.0 - intensity, 1.0  # 파랑 강조: 진할수록 진해짐
    color = mcolors.to_hex((r, g, b))

    folium.CircleMarker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>지역:</b> {row['지역아이디']}<br><b>다발지역지수:</b> {row['count']}",
            max_width=250
        )
    ).add_to(m)

# 4. 지도 확대 자동 조정
m.fit_bounds(df[["centroid_lat", "centroid_lon"]].dropna().values.tolist())

# 5. 지도 출력
m


MULTIPOLYGON (((x1 y1, x2 y2, ..., xn yn))) 형식으로, 지도 위에 다각형 형태로 시각화할 때 사용하는 공간 데이터 그대로 사용했을때 아래 식

In [45]:
# 5. 다발지역지수에서 첫 번째 숫자 추출
df["count"] = df["다발지역지수"].astype(str).str.strip().str.split().str[0]
df["count"] = pd.to_numeric(df["count"], errors="coerce")

# 6. folium 지도 생성
m = folium.Map(tiles="cartodbpositron")

In [46]:
# 8. Polygon 시각화
for _, row in df.iterrows():
    if pd.isnull(row["count"]) or not row["coords"]:
        continue

    intensity = float(norm(row["count"]))
    r, g, b = 1.0 - intensity, 1.0 - intensity, 1.0  # 파랑 강조
    color = mcolors.to_hex((r, g, b))

    folium.Polygon(
        locations=[(y, x) for x, y in row["coords"]],  # (lat, lon) 순서로 변환
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=1.8,
        weight=10,
        popup=folium.Popup(
            f"<b>지역:</b> {row['지역아이디']}<br><b>다발지역지수:</b> {row['count']}",
            max_width=250
        )
    ).add_to(m)

In [47]:
# 9. 전체 다각형 범위에 맞게 확대
all_points = [coord for row in df["coords"] for coord in row]
m.fit_bounds([(y, x) for x, y in all_points])

# 10. 지도 출력
m

In [48]:
# 다발지역지수 기준 내림차순 정렬
sorted_by_count = df[['다발지역지수', '지오메트리']]
sorted_by_count = sorted_by_count.sort_values(by='다발지역지수', ascending=False)

# 결과 출력 (상위 10개) - 지오메트리만 출력
print(sorted_by_count[['지오메트리']].head(10))


                                                                                      지오메트리
60f913f9-21-20211104180334-201 16.666667 16.666667 15  126.859233680581 37.5498524203663)))
b0f913f9-24-20211104180342-167 22.580645 22.580645 35  126.851184538962 37.5642299827022)))
20f913f9-1-20211104180323-183  13.333333 13.333333 10  126.850275353925 37.5385366834414)))
e0f913f9-3-20211104180322-160  12.903226 12.903226 24  126.848526553646 37.5443857850248)))
50f913f9-2-20211104180322-156  12.903226 12.903226 36  126.848522697971 37.5448363961395)))
60f913f9-0-20211104180322-164  12.903226 12.903226 16   126.84795658423 37.5448332529258)))
30f913f9-2-20211104180322-130  45.161290 45.161290 41  126.844612121044 37.5389560300573)))
20f913f9-27-20211104180343-143 45.161290 45.161290 17   126.84460818155 37.5394066830815)))
f0f913f9-22-20211104180344-107 45.161290 45.161290 21  126.844046174942 37.5389528868436)))
d0f913f9-3-20211104180322-102  23.333333 23.333333 54  126.840741861072 37.52856

In [49]:
# 다발지역지수 기준 내림차순 정렬
sorted_by_count = df[['다발지역지수', '지오메트리']]
sorted_by_count = sorted_by_count.sort_values(by='다발지역지수', ascending=False)

# 지오메트리에서 좌표 추출 후 카카오 API 호출하여 주소를 얻고 리스트에 저장
addresses_count = []
for _, row in sorted_by_count.iterrows():
    if row["지오메트리"]:
        coords = extract_coords(row["지오메트리"])
        lat, lon = coords[0]  # 첫 번째 좌표 사용
        address = kakao_reverse_geocode(lat, lon)
        addresses_count.append(address)
    else:
        addresses_count.append("주소 없음")

# 결과를 DataFrame에 저장
sorted_by_count['주소'] = addresses_count
print(sorted_by_count[['지오메트리', '주소']].head(10))


오류 발생 at (126.859233680581, 37.5498524203663): 'documents'
오류 발생 at (126.851184538962, 37.5642299827022): 'documents'
오류 발생 at (126.850275353925, 37.5385366834414): 'documents'
오류 발생 at (126.848526553646, 37.5443857850248): 'documents'
오류 발생 at (126.848522697971, 37.5448363961395): 'documents'
오류 발생 at (126.84795658423, 37.5448332529258): 'documents'
오류 발생 at (126.844612121044, 37.5389560300573): 'documents'
오류 발생 at (126.84460818155, 37.5394066830815): 'documents'
오류 발생 at (126.844046174942, 37.5389528868436): 'documents'
오류 발생 at (126.840741861072, 37.5285690080042): 'documents'
오류 발생 at (126.840175914969, 37.5285658647905): 'documents'
오류 발생 at (126.838454439695, 37.5312602694758): 'documents'
오류 발생 at (126.835513229871, 37.5438625866286): 'documents'
오류 발생 at (126.835441229323, 37.5519742153361): 'documents'
오류 발생 at (126.833185156264, 37.5510602107044): 'documents'
오류 발생 at (126.833181049131, 37.5515108637286): 'documents'
오류 발생 at (126.833177109637, 37.5519614748433): 'documents'

지오메트리를 실제 주소로 변경해야 강서구만 위주로 조사했는지 알수 있다.

In [50]:
# 다발지역지수 기준 내림차순 정렬
sorted_by_count = df[['다발지역지수', '지오메트리']]
sorted_by_count = sorted_by_count.sort_values(by='다발지역지수', ascending=False)

# 지오메트리에서 좌표 추출 후 카카오 API 호출하여 주소를 얻고 리스트에 저장
addresses_count = []
for _, row in sorted_by_count.iterrows():
    if row["지오메트리"]:
        coords = extract_coords(row["지오메트리"])
        lat, lon = coords[0]  # 첫 번째 좌표 사용
        address = kakao_reverse_geocode(lat, lon)
        addresses_count.append(address)
    else:
        addresses_count.append("주소 없음")

# 결과를 DataFrame에 저장
sorted_by_count['주소'] = addresses_count
print(sorted_by_count[['지오메트리', '주소']].head(10))


오류 발생 at (126.859233680581, 37.5498524203663): 'documents'
오류 발생 at (126.851184538962, 37.5642299827022): 'documents'
오류 발생 at (126.850275353925, 37.5385366834414): 'documents'
오류 발생 at (126.848526553646, 37.5443857850248): 'documents'
오류 발생 at (126.848522697971, 37.5448363961395): 'documents'
오류 발생 at (126.84795658423, 37.5448332529258): 'documents'
오류 발생 at (126.844612121044, 37.5389560300573): 'documents'
오류 발생 at (126.84460818155, 37.5394066830815): 'documents'
오류 발생 at (126.844046174942, 37.5389528868436): 'documents'
오류 발생 at (126.840741861072, 37.5285690080042): 'documents'
오류 발생 at (126.840175914969, 37.5285658647905): 'documents'
오류 발생 at (126.838454439695, 37.5312602694758): 'documents'
오류 발생 at (126.835513229871, 37.5438625866286): 'documents'
오류 발생 at (126.835441229323, 37.5519742153361): 'documents'
오류 발생 at (126.833185156264, 37.5510602107044): 'documents'
오류 발생 at (126.833181049131, 37.5515108637286): 'documents'
오류 발생 at (126.833177109637, 37.5519614748433): 'documents'

In [51]:
# 4. 상위 10개 주소 조회
top_df = df.sort_values("count", ascending=False).head(10)
print(top_df)

                                                                                                   지역아이디  \
60f913f9-21-20211104180334-201 16.666667 16.666667 15  MULTIPOLYGON (((126.859233680581 37.5498524203663   
b0f913f9-24-20211104180342-167 22.580645 22.580645 35  MULTIPOLYGON (((126.851184538962 37.5642299827022   
20f913f9-1-20211104180323-183  13.333333 13.333333 10  MULTIPOLYGON (((126.850275353925 37.5385366834414   
e0f913f9-3-20211104180322-160  12.903226 12.903226 24  MULTIPOLYGON (((126.848526553646 37.5443857850248   
50f913f9-2-20211104180322-156  12.903226 12.903226 36  MULTIPOLYGON (((126.848522697971 37.5448363961395   
60f913f9-0-20211104180322-164  12.903226 12.903226 16   MULTIPOLYGON (((126.84795658423 37.5448332529258   
30f913f9-2-20211104180322-130  45.161290 45.161290 41  MULTIPOLYGON (((126.844612121044 37.5389560300573   
20f913f9-27-20211104180343-143 45.161290 45.161290 17   MULTIPOLYGON (((126.84460818155 37.5394066830815   
f0f913f9-22-20211104180344-1

In [52]:
results = []
for _, row in top_df.iterrows():
    lat = row.get("centroid_lat")
    lon = row.get("centroid_lon")
    if pd.isnull(lat) or pd.isnull(lon):
        continue
    address = kakao_reverse_geocode(lat, lon)
    results.append((row["지역아이디"], row["count"], address))


오류 발생 at (37.5498524203663, 126.859233680581): 'documents'
오류 발생 at (37.5642299827022, 126.851184538962): 'documents'
오류 발생 at (37.5385366834414, 126.850275353925): 'documents'
오류 발생 at (37.5443857850248, 126.848526553646): 'documents'
오류 발생 at (37.5448363961395, 126.848522697971): 'documents'
오류 발생 at (37.5448332529258, 126.84795658423): 'documents'
오류 발생 at (37.5389560300573, 126.844612121044): 'documents'
오류 발생 at (37.5394066830815, 126.84460818155): 'documents'
오류 발생 at (37.5389528868436, 126.844046174942): 'documents'
오류 발생 at (37.5285690080042, 126.840741861072): 'documents'
